In [ ]:
# run this once to install the required packages
%pip install pandas
%pip install numpy
%pip install matplotlib
%pip install statsmodels
%pip install scipy

# Current version: 2.9.2 (2024-06-03)
%pip install --upgrade "git+https://github.com/science64/DynaTMT-py-SB.git"

# Current version: 2.1.2 (2025-11-05)
%pip install --upgrade "git+https://github.com/science64/PBLMM.git"

In [3]:
# import the required packages
from datetime import date
import pandas as pd
import warnings
import DynaTMT_SB.DynaTMT as mePROD
import PBLMM

warnings.filterwarnings("ignore")

In [4]:
wd = "Example data/MS3_data" # you can define your folder here etc: C://Users/User/Data/fractionation/

nameOfStudy = "48h_siUSP39_DB" # please define a name for your study

dataName = "20240109_LU_LC2_MAA_DB_mePROD_MS3_PSMs.txt" # please define the name of your data file (PSMs) here

conditions = ['Cont', 'Cont', 'Cont', 'USP39', 'USP39', 'USP39'] # define the conditions of TMT multiplexing here
pairs = [['USP39', 'Cont']] # define the pairs of conditions you want to compare here. result will be log2(USP39/Cont)

In [5]:
psms = pd.read_csv(f'{wd}/{dataName}', sep='\t', header=0) # TEXT or CSV file: you provide your .txt PSM or peptide file here.

boster_removed = psms.drop('Abundance 131C', axis = True) # remove the booster channel if present  (location of the booster Abundance: 131C channel in the data file)
baseline_removed = boster_removed.drop('Abundance 126', axis = True) # remove the baseline channel (location of the baseline Abundance: 126 channel in the data file)

In [6]:
process = mePROD.PD_input(baseline_removed) # initiate your date here with PD_input class, if your data name is 'baseline_removed'

filter_data = process.filter_PSMs(baseline_removed) # filter contamination, NA samples, shared peptides

sumNorm = process.total_intensity_normalisation(filter_data) # for total intenstiy normalization

heavy = process.extract_heavy(sumNorm) # extract heavy PSMs/peptides

light = process.extract_light(sumNorm) # extract light PSMs/peptides (OPTIONAL)

peptide_data = process.PSMs_to_Peptide(heavy) # convert PSMs to Peptide level data

# PBLMM analysis ==> this is the main part of the statistical analysis based on peptide based linear mixed model (LMM)
hypothesis_testing = PBLMM.HypothesisTesting()
resultFinal = hypothesis_testing.peptide_based_lmm(peptide_data,conditions=conditions,pairs=pairs)
resultFinal.reset_index(inplace=True)
resultFinal.rename(columns={'index': 'Accession'}, inplace=True)

resultFinal.to_excel(f'{wd}/{nameOfStudy}_mePROD_PBLMM_MS3_{date.today().strftime("%d.%m.%Y")}.xlsx', index=False, engine='openpyxl')

print('[#] COMPLETED: resultFinal: %s rows x %s columns' % (resultFinal.shape[0], resultFinal.shape[1]))

Calling function: filter_PSMs
Calling function: total_intensity_normalisation
Calling function: extract_heavy
Extraction Done Extracted Heavy PSMs/Peptides: 22928
Calling function: extract_light
Extraction Done Extracted Light PSMs/Peptides: 60807
Calling function: PSMs_to_Peptide
Calculate Protein quantifications from Peptides
Combination done
Total Number of Datapoints:  89886
['USP39', 'Cont'] is analysing via PBLMM...
[#] COMPLETED: resultFinal: 3615 rows x 10 columns
